# Version 2 Model Run and Review



## Mount Drive and Set Root

Purpose: mount Google Drive in Colab, then point the notebook to the model project root.

In [9]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd
from IPython.display import display

root_candidates = [
    Path('/content/drive/MyDrive/v2/model'),
    Path('/content/v2/model'),
    Path.cwd().resolve().parents[0],
]
ROOT = next(path for path in root_candidates if path.exists())
AVAILABLE_PROFILES = ['careful_v3', 'max_v3']
DEFAULT_FEATURE_PROFILE = 'careful_v3'
print({'ROOT': str(ROOT), 'available_profiles': AVAILABLE_PROFILES, 'default_profile': DEFAULT_FEATURE_PROFILE})
ROOT

{'ROOT': '/content/drive/MyDrive/v2/model', 'available_profiles': ['careful_v3', 'max_v3'], 'default_profile': 'careful_v3'}


PosixPath('/content/drive/MyDrive/v2/model')

## Install Requirements and Load Helpers

Purpose: install dependencies, load helper functions, and support running each model under both `careful_v3` and `max_v3` from the same notebook.

In [11]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from v2_model.config import load_config
from v2_model.recommend import build_latest_recommendations
from v2_model.recommend_true_latest import build_true_latest_recommendations

LAST_RUN_DIRS = {}

def _config_for_profile(feature_profile: str):
    cfg = ROOT / 'configs' / f'{feature_profile}.yaml'
    if cfg.exists():
        return cfg
    return ROOT / 'configs' / 'default.yaml'

def _run_key(model_name: str, feature_profile: str):
    return (model_name.upper(), feature_profile)

def _parse_run_dir(stdout_text: str):
    marker = 'Pipeline completed. Outputs saved to:'
    for line in reversed(stdout_text.splitlines()):
        if marker in line:
            return Path(line.split(marker, 1)[1].strip())
    return None

def _manifest_feature_profile(run_dir: Path):
    manifest = run_dir / 'meta' / 'run_manifest.json'
    if not manifest.exists():
        return None
    try:
        return json.loads(manifest.read_text()).get('feature_profile')
    except Exception:
        return None

def run_single_model(model_name: str, feature_profile: str = DEFAULT_FEATURE_PROFILE, stages: str = 'all'):
    config_path = _config_for_profile(feature_profile)
    cmd = [sys.executable, str(ROOT / 'run_model.py'), '--config', str(config_path), '--models', model_name, '--stages', stages]
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line.rstrip(''))
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Model run failed with code {rc}')
    run_dir = _parse_run_dir(''.join(lines))
    if run_dir is None:
        raise RuntimeError('Could not parse run directory from command output.')
    LAST_RUN_DIRS[_run_key(model_name, feature_profile)] = run_dir
    return run_dir

def get_run_dir(model_name: str, feature_profile: str = DEFAULT_FEATURE_PROFILE):
    key = _run_key(model_name, feature_profile)
    if key in LAST_RUN_DIRS:
        return LAST_RUN_DIRS[key]
    candidates = sorted((ROOT / 'outputs').glob('run_*'))
    model_key = model_name.lower()
    for run_dir in reversed(candidates):
        if not (run_dir / 'r2' / f'{model_key}_r2_summary_full_large_small.csv').exists():
            continue
        if _manifest_feature_profile(run_dir) != feature_profile:
            continue
        LAST_RUN_DIRS[key] = run_dir
        return run_dir
    raise FileNotFoundError(f'No saved run found for {model_name.upper()} with profile {feature_profile}. Run the model first.')

def show_model_summary(model_name: str, feature_profile: str = DEFAULT_FEATURE_PROFILE):
    key = model_name.lower()
    run_dir = get_run_dir(model_name, feature_profile)
    print('Feature profile:', feature_profile)
    print('Run dir:', run_dir)
    display(pd.read_csv(run_dir / 'preprocess' / 'panel_prep_summary.csv'))
    display(pd.read_csv(run_dir / 'preprocess' / 'window_coverage_summary.csv'))
    display(pd.read_csv(run_dir / 'preprocess' / 'preprocess_report.csv'))
    display(pd.read_csv(run_dir / 'r2' / f'{key}_r2_summary_full_large_small.csv'))
    display(pd.read_csv(run_dir / 'portfolio' / f'{key}_performance_summary.csv'))
    display(pd.read_csv(run_dir / 'benchmark' / f'{key}_vs_benchmark.csv'))
    imp_path = run_dir / 'importance' / f'{key}_feature_importance.csv'
    if imp_path.exists():
        display(pd.read_csv(imp_path).head(15))
    comp_path = run_dir / 'complexity' / f'{key}_complexity.csv'
    if comp_path.exists():
        display(pd.read_csv(comp_path).head(15))

def show_latest_recommendations(model_name: str, feature_profile: str = DEFAULT_FEATURE_PROFILE, top_k: int = 10, save_to_run_dir: bool = True):
    cfg = load_config(_config_for_profile(feature_profile))
    result = build_latest_recommendations(cfg, model_name, top_k=top_k)
    print('Feature profile:', feature_profile)
    print('Latest fully labeled month scored:', result.latest_eom.date())
    print('Calibration window train:', result.train_start.date(), '->', result.train_end.date())
    print('Calibration window val  :', result.val_start.date(), '->', result.val_end.date())
    display(result.recommendations)
    if save_to_run_dir:
        run_dir = get_run_dir(model_name, feature_profile)
        out_path = run_dir / 'recommendations' / f"{model_name.lower()}_{feature_profile}_latest_recommendations.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        result.recommendations.to_csv(out_path, index=False)
        print('Saved:', out_path)

def show_true_latest_recommendations(model_name: str, feature_profile: str = DEFAULT_FEATURE_PROFILE, top_k: int = 10, save_to_run_dir: bool = True):
    cfg = load_config(_config_for_profile(feature_profile))
    result = build_true_latest_recommendations(cfg, model_name, top_k=top_k)
    print('Feature profile:', feature_profile)
    print('True latest month scored:', result.score_eom.date())
    print('Latest labeled month used for calibration:', result.latest_labeled_eom.date())
    print('Calibration window train:', result.train_start.date(), '->', result.train_end.date())
    print('Calibration window val  :', result.val_start.date(), '->', result.val_end.date())
    print('Eligible latest-month rows:', result.universe_rows)
    print('Scored latest-month rows  :', result.scored_rows)
    display(result.recommendations)
    if save_to_run_dir:
        run_dir = get_run_dir(model_name, feature_profile)
        out_path = run_dir / 'recommendations' / f"{model_name.lower()}_{feature_profile}_true_latest_recommendations.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        result.recommendations.to_csv(out_path, index=False)
        print('Saved:', out_path)


## OLS

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [12]:
RUN_DIR_OLS_CAREFUL = run_single_model('OLS', feature_profile='careful_v3')
RUN_DIR_OLS_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models OLS --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWar

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T194616Z')

In [13]:
show_model_summary('OLS', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T194616Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_OLS,n_rows,n_months
0,Full sample,-0.024925,12716,156
1,Large firms,-0.027424,3756,156
2,Small firms,-0.116210,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,OLS_LS_EW_gross,1.938802,0,156,0.039034,0.066822,0.544741,0.231477,2.023552,284.184180,-0.196799,-0.178979
1,OLS_LS_EW_net_0bps,1.938802,0,156,0.039034,0.066822,0.544741,0.231477,2.023552,284.184180,-0.196799,-0.178979
2,OLS_LS_EW_net_10bps,1.938802,10,156,0.037095,0.066684,0.510531,0.231001,1.927006,212.150366,-0.198962,-0.180446
3,OLS_LS_EW_net_20bps,1.938802,20,156,0.035156,0.066554,0.477009,0.230549,1.829871,158.214096,-0.201123,-0.181912
4,OLS_LS_EW_net_30bps,1.938802,30,156,0.033217,0.066430,0.444163,0.230120,1.732174,117.853097,-0.203281,-0.183379
5,OLS_LS_VW_gross,2.511225,0,156,0.032015,0.089522,0.393306,0.310113,1.238839,73.576971,-0.419076,-0.364238
6,OLS_LS_VW_net_0bps,2.511225,0,156,0.032015,0.089522,0.393306,0.310113,1.238839,73.576971,-0.419076,-0.364238
7,OLS_LS_VW_net_10bps,2.511225,10,156,0.029504,0.089561,0.352803,0.310248,1.141170,49.821567,-0.457458,-0.366234
8,OLS_LS_VW_net_20bps,2.511225,20,156,0.026993,0.089609,0.313368,0.310413,1.043481,33.595944,-0.493407,-0.368230
9,OLS_LS_VW_net_30bps,2.511225,30,156,0.024481,0.089666,0.274978,0.310611,0.945801,22.525200,-0.529914,-0.370227


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,OLS_LS_EW_net_30bps,benchmark_equal_excess,156,0.025557,0.289955,-0.011149
1,value,OLS_LS_VW_net_30bps,benchmark_value_excess,156,0.016820,0.165258,0.092727


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.011050,0.001918,0.215719
1,Vol_90D,0.011283,0.001684,0.189447
2,mom36m,0.012040,0.000928,0.104365
3,lev,0.012244,0.000724,0.081391
4,be_me,0.012306,0.000661,0.074359
5,Vol_30D,0.012450,0.000517,0.058181
6,roeq,0.012614,0.000354,0.039812
7,me,0.012708,0.000260,0.029203
8,maxret,0.012786,0.000182,0.020446
9,cash_ratio,0.012788,0.000179,0.020192


,window_id,test_start,test_end,best_score,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.126695,1000,55
1,2,2013-03-31,2014-02-28,0.129978,1000,55
2,3,2014-03-31,2015-02-28,0.131374,1000,55
3,4,2015-03-31,2016-02-29,0.118757,1000,55
4,5,2016-03-31,2017-02-28,0.113906,1000,55
5,6,2017-03-31,2018-02-28,0.133433,1000,55
6,7,2018-03-31,2019-02-28,0.144182,1000,55
7,8,2019-03-31,2020-02-29,0.129536,1000,55
8,9,2020-03-31,2021-02-28,0.122215,1000,55
9,10,2021-03-31,2022-02-28,0.154672,1000,55


In [14]:
show_latest_recommendations('OLS', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TDC VM Equity,2026-02-28,0.025906,11500.0000,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
1,2,TCD VM Equity,2026-02-28,0.025094,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
2,3,BCG VM Equity,2026-02-28,0.020073,2530.0000,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
3,4,BAB VH Equity,2026-02-28,0.019436,12000.0000,1.286568e+07,1.146298e+08,8.636930,-0.035759,-0.139302,0.069358,1.034450
4,5,AAT VM Equity,2026-02-28,0.019351,3140.0000,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
5,6,AAA VM Equity,2026-02-28,0.015584,7900.0000,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
6,7,AFX VM Equity,2026-02-28,0.015524,10700.0000,3.745000e+05,1.588130e+09,3388.571429,-0.036036,NaN,NaN,1.319599
7,8,GDT VM Equity,2026-02-28,0.014149,492184.2755,5.456466e+05,1.373137e+10,1.382804,0.010127,-0.036232,-0.143410,0.652403
8,9,TCL VM Equity,2026-02-28,0.013868,35100.0000,1.058561e+06,7.072450e+08,308.190311,-0.011268,0.001427,-0.169231,0.649413
9,10,APH VM Equity,2026-02-28,0.013441,6220.0000,1.516960e+06,1.860843e+09,500.237264,-0.038640,-0.086637,-0.143251,1.667269


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T194616Z/recommendations/ols_careful_v3_latest_recommendations.csv


### max_v3

In [15]:
RUN_DIR_OLS_MAX = run_single_model('OLS', feature_profile='max_v3')
RUN_DIR_OLS_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models OLS --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T194948Z')

In [16]:
show_model_summary('OLS', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T194948Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_OLS,n_rows,n_months
0,Full sample,-0.028837,12716,156
1,Large firms,-0.087287,3756,156
2,Small firms,-0.146826,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,OLS_LS_EW_gross,2.085231,0,156,0.034380,0.075160,0.454210,0.260363,1.584541,129.062954,-0.162376,-0.160157
1,OLS_LS_EW_net_0bps,2.085231,0,156,0.034380,0.075160,0.454210,0.260363,1.584541,129.062954,-0.162376,-0.160157
2,OLS_LS_EW_net_10bps,2.085231,10,156,0.032294,0.075049,0.419377,0.259977,1.490646,93.900950,-0.174436,-0.162157
3,OLS_LS_EW_net_20bps,2.085231,20,156,0.030209,0.074944,0.385302,0.259613,1.396351,68.195739,-0.194819,-0.164157
4,OLS_LS_EW_net_30bps,2.085231,30,156,0.028124,0.074845,0.351972,0.259271,1.301679,49.417170,-0.218822,-0.166157
5,OLS_LS_VW_gross,2.538254,0,156,0.027728,0.101292,0.310551,0.350885,0.948291,32.643433,-0.300477,-0.264543
6,OLS_LS_VW_net_0bps,2.538254,0,156,0.027728,0.101292,0.310551,0.350885,0.948291,32.643433,-0.300477,-0.264543
7,OLS_LS_VW_net_10bps,2.538254,10,156,0.025190,0.101235,0.271929,0.350688,0.861969,21.804385,-0.311273,-0.265311
8,OLS_LS_VW_net_20bps,2.538254,20,156,0.022652,0.101187,0.234340,0.350522,0.775482,14.440131,-0.321945,-0.266079
9,OLS_LS_VW_net_30bps,2.538254,30,156,0.020114,0.101148,0.197757,0.350386,0.688852,9.442274,-0.344166,-0.266848


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,OLS_LS_EW_net_30bps,benchmark_equal_excess,156,0.020463,0.212963,-0.042070
1,value,OLS_LS_VW_net_30bps,benchmark_value_excess,156,0.012453,0.107540,0.008009


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.016445,0.001337,0.141196
1,ret_12_1_x_vn_mkt_chg1m,0.016836,0.000946,0.099938
2,cur_assets_assets,0.017097,0.000685,0.072367
3,ebitda_assets_raw,0.017179,0.000603,0.063699
4,cur_liab_assets,0.017183,0.000599,0.063283
5,Vol_90D,0.017378,0.000404,0.042657
6,hl_range_avg_1m,0.017390,0.000392,0.041403
7,ebitda_growth_12m_raw,0.017430,0.000352,0.037167
8,receivables_assets,0.017488,0.000294,0.031081
9,tri_mom36m,0.017490,0.000292,0.030824


,window_id,test_start,test_end,best_score,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.134153,1000,134
1,2,2013-03-31,2014-02-28,0.130606,1000,134
2,3,2014-03-31,2015-02-28,0.131348,1000,134
3,4,2015-03-31,2016-02-29,0.120007,1000,134
4,5,2016-03-31,2017-02-28,0.117298,1000,134
5,6,2017-03-31,2018-02-28,0.133758,1000,134
6,7,2018-03-31,2019-02-28,0.144369,1000,134
7,8,2019-03-31,2020-02-29,0.129789,1000,134
8,9,2020-03-31,2021-02-28,0.122246,1000,134
9,10,2021-03-31,2022-02-28,0.154968,1000,134


In [17]:
show_latest_recommendations('OLS', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,0.032959,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,BCC VH Equity,2026-02-28,0.031851,7800.0000,9.610365e+05,5.184796e+08,254.922883,-0.025000,-0.071429,-0.037037,1.918663
2,3,BAB VH Equity,2026-02-28,0.024476,12000.0000,1.286568e+07,1.146298e+08,8.636930,-0.035759,-0.139302,0.069358,1.034450
3,4,AAT VM Equity,2026-02-28,0.023416,3140.0000,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
4,5,SMC VM Equity,2026-02-28,0.023000,12150.0000,8.943231e+05,4.008225e+09,10546.411312,-0.122744,0.029661,1.018272,0.750421
5,6,GDT VM Equity,2026-02-28,0.020729,492184.2755,5.456466e+05,1.373137e+10,1.382804,0.010127,-0.036232,-0.143410,0.652403
6,7,API VH Equity,2026-02-28,0.020092,6300.0000,5.297290e+05,1.016526e+09,776.604533,-0.045455,-0.231707,-0.171053,1.702812
7,8,AFX VM Equity,2026-02-28,0.019690,10700.0000,3.745000e+05,1.588130e+09,3388.571429,-0.036036,NaN,NaN,1.319599
8,9,AGG VM Equity,2026-02-28,0.018182,14400.0000,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
9,10,DXP VH Equity,2026-02-28,0.016559,772840.7157,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T194948Z/recommendations/ols_max_v3_latest_recommendations.csv


## OLS3

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [18]:
RUN_DIR_OLS3_CAREFUL = run_single_model('OLS3', feature_profile='careful_v3')
RUN_DIR_OLS3_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models OLS3 --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWa

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T202107Z')

In [19]:
show_model_summary('OLS3', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T202107Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_OLS3,n_rows,n_months
0,Full sample,-0.011397,12716,156
1,Large firms,-0.020293,3756,156
2,Small firms,-0.006599,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,OLS3_LS_EW_gross,1.133406,0,156,0.014588,0.083150,0.142275,0.288041,0.607760,4.636625,-0.467785,-0.217652
1,OLS3_LS_EW_net_0bps,1.133406,0,156,0.014588,0.083150,0.142275,0.288041,0.607760,4.636625,-0.467785,-0.217652
2,OLS3_LS_EW_net_10bps,1.133406,10,156,0.013455,0.083184,0.126914,0.288159,0.560313,3.726943,-0.481883,-0.218223
3,OLS3_LS_EW_net_20bps,1.133406,20,156,0.012322,0.083221,0.111739,0.288287,0.512886,2.963123,-0.495630,-0.219313
4,OLS3_LS_EW_net_30bps,1.133406,30,156,0.011188,0.083261,0.096748,0.288425,0.465484,2.321929,-0.509033,-0.220904
5,OLS3_LS_VW_gross,1.477180,0,156,0.006409,0.087018,0.031842,0.301441,0.255129,0.503046,-0.756577,-0.213425
6,OLS3_LS_VW_net_0bps,1.477180,0,156,0.006409,0.087018,0.031842,0.301441,0.255129,0.503046,-0.756577,-0.213425
7,OLS3_LS_VW_net_10bps,1.477180,10,156,0.004932,0.087168,0.013523,0.301960,0.195987,0.190797,-0.786904,-0.214264
8,OLS3_LS_VW_net_20bps,1.477180,20,156,0.003455,0.087325,-0.004506,0.302504,0.137036,-0.057020,-0.813714,-0.215104
9,OLS3_LS_VW_net_30bps,1.477180,30,156,0.001977,0.087490,-0.022249,0.303073,0.078291,-0.253610,-0.837197,-0.215943


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,OLS3_LS_EW_net_30bps,benchmark_equal_excess,156,0.003527,0.036576,0.094850
1,value,OLS3_LS_VW_net_30bps,benchmark_value_excess,156,-0.005684,-0.059632,0.184016


,Feature,R2OOS,red_R2OOS,var_imp
0,be_me,-0.000072,0.002113,0.807885
1,me,0.001645,0.000395,0.151229
2,ret_12_1,0.001934,0.000107,0.040886


,window_id,test_start,test_end,best_score,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.125012,1000,3
1,2,2013-03-31,2014-02-28,0.129101,1000,3
2,3,2014-03-31,2015-02-28,0.131273,1000,3
3,4,2015-03-31,2016-02-29,0.116814,1000,3
4,5,2016-03-31,2017-02-28,0.108196,1000,3
5,6,2017-03-31,2018-02-28,0.134744,1000,3
6,7,2018-03-31,2019-02-28,0.143847,1000,3
7,8,2019-03-31,2020-02-29,0.129665,1000,3
8,9,2020-03-31,2021-02-28,0.124063,1000,3
9,10,2021-03-31,2022-02-28,0.154131,1000,3


In [20]:
show_latest_recommendations('OLS3', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,BCG VM Equity,2026-02-28,0.008282,2.530000e+03,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
1,2,AAA VM Equity,2026-02-28,0.008084,7.900000e+03,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
2,3,ASM VM Equity,2026-02-28,0.007943,6.300000e+03,2.565323e+06,2.360878e+09,1123.788164,-0.021739,-0.147601,-0.157959,2.135306
3,4,TCB VM Equity,2026-02-28,0.007771,3.625000e+04,2.568762e+08,3.708119e+11,102852.158180,0.006944,-0.070513,0.367925,0.661971
4,5,ACB VM Equity,2026-02-28,0.007013,2.455000e+04,1.261049e+08,3.049721e+11,2151.399414,-0.012072,-0.118492,0.077576,0.749532
5,6,BAB VH Equity,2026-02-28,0.006744,1.200000e+04,1.286568e+07,1.146298e+08,8.636930,-0.035759,-0.139302,0.069358,1.034450
6,7,SSB VM Equity,2026-02-28,0.006204,1.690000e+04,4.808050e+07,3.908479e+10,22540.054178,-0.045198,-0.235294,-0.135550,0.839687
7,8,AGG VM Equity,2026-02-28,0.005678,1.440000e+04,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
8,9,TCH VM Equity,2026-02-28,0.005417,1.520000e+04,1.386406e+07,9.435737e+10,134641.029545,-0.052960,-0.223358,0.078405,0.820794
9,10,VRE VM Equity,2026-02-28,0.005306,6.601085e+07,7.829674e+07,9.570442e+08,0.000444,-0.087912,-0.031667,0.684058,0.617755


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T202107Z/recommendations/ols3_careful_v3_latest_recommendations.csv


### max_v3

In [21]:
RUN_DIR_OLS3_MAX = run_single_model('OLS3', feature_profile='max_v3')
RUN_DIR_OLS3_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models OLS3 --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarnin

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T202359Z')

In [22]:
show_model_summary('OLS3', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T202359Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_OLS3,n_rows,n_months
0,Full sample,-0.011397,12716,156
1,Large firms,-0.020293,3756,156
2,Small firms,-0.006599,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,OLS3_LS_EW_gross,1.133406,0,156,0.014588,0.083150,0.142275,0.288041,0.607760,4.636625,-0.467785,-0.217652
1,OLS3_LS_EW_net_0bps,1.133406,0,156,0.014588,0.083150,0.142275,0.288041,0.607760,4.636625,-0.467785,-0.217652
2,OLS3_LS_EW_net_10bps,1.133406,10,156,0.013455,0.083184,0.126914,0.288159,0.560313,3.726943,-0.481883,-0.218223
3,OLS3_LS_EW_net_20bps,1.133406,20,156,0.012322,0.083221,0.111739,0.288287,0.512886,2.963123,-0.495630,-0.219313
4,OLS3_LS_EW_net_30bps,1.133406,30,156,0.011188,0.083261,0.096748,0.288425,0.465484,2.321929,-0.509033,-0.220904
5,OLS3_LS_VW_gross,1.477180,0,156,0.006409,0.087018,0.031842,0.301441,0.255129,0.503046,-0.756577,-0.213425
6,OLS3_LS_VW_net_0bps,1.477180,0,156,0.006409,0.087018,0.031842,0.301441,0.255129,0.503046,-0.756577,-0.213425
7,OLS3_LS_VW_net_10bps,1.477180,10,156,0.004932,0.087168,0.013523,0.301960,0.195987,0.190797,-0.786904,-0.214264
8,OLS3_LS_VW_net_20bps,1.477180,20,156,0.003455,0.087325,-0.004506,0.302504,0.137036,-0.057020,-0.813714,-0.215104
9,OLS3_LS_VW_net_30bps,1.477180,30,156,0.001977,0.087490,-0.022249,0.303073,0.078291,-0.253610,-0.837197,-0.215943


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,OLS3_LS_EW_net_30bps,benchmark_equal_excess,156,0.003527,0.036576,0.094850
1,value,OLS3_LS_VW_net_30bps,benchmark_value_excess,156,-0.005684,-0.059632,0.184016


,Feature,R2OOS,red_R2OOS,var_imp
0,be_me,-0.000072,0.002113,0.807885
1,me,0.001645,0.000395,0.151229
2,ret_12_1,0.001934,0.000107,0.040886


,window_id,test_start,test_end,best_score,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.125012,1000,3
1,2,2013-03-31,2014-02-28,0.129101,1000,3
2,3,2014-03-31,2015-02-28,0.131273,1000,3
3,4,2015-03-31,2016-02-29,0.116814,1000,3
4,5,2016-03-31,2017-02-28,0.108196,1000,3
5,6,2017-03-31,2018-02-28,0.134744,1000,3
6,7,2018-03-31,2019-02-28,0.143847,1000,3
7,8,2019-03-31,2020-02-29,0.129665,1000,3
8,9,2020-03-31,2021-02-28,0.124063,1000,3
9,10,2021-03-31,2022-02-28,0.154131,1000,3


In [23]:
show_latest_recommendations('OLS3', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,BCG VM Equity,2026-02-28,0.008282,2.530000e+03,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
1,2,AAA VM Equity,2026-02-28,0.008084,7.900000e+03,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
2,3,ASM VM Equity,2026-02-28,0.007943,6.300000e+03,2.565323e+06,2.360878e+09,1123.788164,-0.021739,-0.147601,-0.157959,2.135306
3,4,TCB VM Equity,2026-02-28,0.007771,3.625000e+04,2.568762e+08,3.708119e+11,102852.158180,0.006944,-0.070513,0.367925,0.661971
4,5,ACB VM Equity,2026-02-28,0.007013,2.455000e+04,1.261049e+08,3.049721e+11,2151.399414,-0.012072,-0.118492,0.077576,0.749532
5,6,BAB VH Equity,2026-02-28,0.006744,1.200000e+04,1.286568e+07,1.146298e+08,8.636930,-0.035759,-0.139302,0.069358,1.034450
6,7,SSB VM Equity,2026-02-28,0.006204,1.690000e+04,4.808050e+07,3.908479e+10,22540.054178,-0.045198,-0.235294,-0.135550,0.839687
7,8,AGG VM Equity,2026-02-28,0.005678,1.440000e+04,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
8,9,TCH VM Equity,2026-02-28,0.005417,1.520000e+04,1.386406e+07,9.435737e+10,134641.029545,-0.052960,-0.223358,0.078405,0.820794
9,10,VRE VM Equity,2026-02-28,0.005306,6.601085e+07,7.829674e+07,9.570442e+08,0.000444,-0.087912,-0.031667,0.684058,0.617755


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T202359Z/recommendations/ols3_max_v3_latest_recommendations.csv


## ENET

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [24]:
RUN_DIR_ENET_CAREFUL = run_single_model('ENET', feature_profile='careful_v3')
RUN_DIR_ENET_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models ENET --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWa

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T202730Z')

In [25]:
show_model_summary('ENET', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T202730Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_ENET,n_rows,n_months
0,Full sample,-0.013508,12716,156
1,Large firms,-0.014003,3756,156
2,Small firms,-0.005984,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,ENET_LS_EW_gross,1.482440,0,156,0.038224,0.060577,0.537363,0.209844,2.185857,266.973742,-0.115267,-0.115267
1,ENET_LS_EW_net_0bps,1.482440,0,156,0.038224,0.060577,0.537363,0.209844,2.185857,266.973742,-0.115267,-0.115267
2,ENET_LS_EW_net_10bps,1.482440,10,156,0.036742,0.060337,0.511374,0.209014,2.109427,213.701753,-0.116733,-0.116733
3,ENET_LS_EW_net_20bps,1.482440,20,156,0.035259,0.060109,0.485777,0.208223,2.032010,170.948704,-0.119831,-0.118200
4,ENET_LS_EW_net_30bps,1.482440,30,156,0.033777,0.059892,0.460568,0.207471,1.953635,136.651640,-0.125267,-0.119667
5,ENET_LS_VW_gross,1.867356,0,156,0.030713,0.087932,0.376398,0.304604,1.209942,62.631779,-0.279984,-0.197235
6,ENET_LS_VW_net_0bps,1.867356,0,156,0.030713,0.087932,0.376398,0.304604,1.209942,62.631779,-0.279984,-0.197235
7,ENET_LS_VW_net_10bps,1.867356,10,156,0.028845,0.087838,0.346685,0.304280,1.137589,46.913524,-0.302046,-0.198630
8,ENET_LS_VW_net_20bps,1.867356,20,156,0.026978,0.087757,0.317543,0.304000,1.064925,35.053176,-0.329969,-0.200024
9,ENET_LS_VW_net_30bps,1.867356,30,156,0.025111,0.087689,0.288964,0.303765,0.991981,26.109945,-0.356840,-0.201419


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,ENET_LS_EW_net_30bps,benchmark_equal_excess,156,0.026116,0.312180,-0.020510
1,value,ENET_LS_VW_net_30bps,benchmark_value_excess,156,0.017450,0.161003,-0.078397


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.010545,3.136912e-03,3.949391e-01
1,mom36m,0.011236,2.445132e-03,3.078436e-01
2,Vol_30D,0.011656,2.025529e-03,2.550153e-01
3,be_me,0.012714,9.677060e-04,1.218347e-01
4,lev,0.012819,8.623750e-04,1.085735e-01
5,cash_ratio,0.013595,8.646683e-05,1.088622e-02
6,std_turn,0.013682,1.238670e-10,1.559493e-08
7,Free_Float_Pct,0.013682,0.000000e+00,0.000000e+00
8,Global_Baltic_Dry,0.013682,0.000000e+00,0.000000e+00
9,Hong_Kong_Index,0.013682,0.000000e+00,0.000000e+00


,window_id,test_start,test_end,best_score,best_alpha,best_l1_ratio,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.126259,0.00400,0.5,10000,8
1,2,2013-03-31,2014-02-28,0.125667,0.00400,0.5,10000,6
2,3,2014-03-31,2015-02-28,0.126940,0.00400,0.5,10000,4
3,4,2015-03-31,2016-02-29,0.113407,0.00400,0.5,10000,4
4,5,2016-03-31,2017-02-28,0.107382,0.00400,0.5,10000,6
5,6,2017-03-31,2018-02-28,0.133655,0.00400,0.5,10000,5
6,7,2018-03-31,2019-02-28,0.143070,0.00127,0.5,10000,16
7,8,2019-03-31,2020-02-29,0.130180,0.00379,0.5,10000,6
8,9,2020-03-31,2021-02-28,0.125413,0.00001,0.5,10000,55
9,10,2021-03-31,2022-02-28,0.152187,0.00085,0.5,10000,19


In [26]:
show_latest_recommendations('ENET', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,0.034693,1.890000e+03,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,BCG VM Equity,2026-02-28,0.028668,2.530000e+03,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
2,3,ASP VM Equity,2026-02-28,0.027311,5.020000e+03,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117
3,4,FMC VM Equity,2026-02-28,0.026352,2.811722e+06,1.909705e+06,4.540653e+11,4.464135,0.131579,0.162162,-0.084132,1.230356
4,5,TDC VM Equity,2026-02-28,0.026350,1.150000e+04,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
5,6,TD6 VH Equity,2026-02-28,0.026106,7.200000e+03,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
6,7,DXP VH Equity,2026-02-28,0.025461,7.728407e+05,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881
7,8,DVM VH Equity,2026-02-28,0.024173,3.011698e+05,6.628942e+05,7.570968e+09,4.488833,0.185185,-0.035619,-0.181397,1.170660
8,9,FDC VM Equity,2026-02-28,0.024129,7.145345e+05,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
9,10,DTD VH Equity,2026-02-28,0.024078,1.180343e+06,6.557330e+05,1.106520e+11,5.620928,0.000000,-0.123762,-0.103419,2.025794


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T202730Z/recommendations/enet_careful_v3_latest_recommendations.csv


### max_v3

In [27]:
RUN_DIR_ENET_MAX = run_single_model('ENET', feature_profile='max_v3')
RUN_DIR_ENET_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models ENET --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarnin

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T203049Z')

In [28]:
show_model_summary('ENET', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T203049Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_ENET,n_rows,n_months
0,Full sample,-0.011926,12716,156
1,Large firms,-0.018746,3756,156
2,Small firms,-0.010171,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,ENET_LS_EW_gross,1.523277,0,156,0.043325,0.070144,0.620796,0.242987,2.139610,531.682343,-0.200449,-0.122401
1,ENET_LS_EW_net_0bps,1.523277,0,156,0.043325,0.070144,0.620796,0.242987,2.139610,531.682343,-0.200449,-0.122401
2,ENET_LS_EW_net_10bps,1.523277,10,156,0.041802,0.069851,0.592870,0.241971,2.073050,423.956567,-0.203755,-0.122845
3,ENET_LS_EW_net_20bps,1.523277,20,156,0.040278,0.069569,0.565372,0.240993,2.005613,337.867044,-0.207054,-0.123290
4,ENET_LS_EW_net_30bps,1.523277,30,156,0.038755,0.069297,0.538297,0.240054,1.937315,269.098265,-0.210345,-0.123734
5,ENET_LS_VW_gross,1.914657,0,156,0.050765,0.138847,0.675234,0.480980,1.266543,817.429007,-0.250578,-0.156929
6,ENET_LS_VW_net_0bps,1.914657,0,156,0.050765,0.138847,0.675234,0.480980,1.266543,817.429007,-0.250578,-0.156929
7,ENET_LS_VW_net_10bps,1.914657,10,156,0.048850,0.138732,0.638928,0.480583,1.219779,614.572703,-0.254019,-0.159537
8,ENET_LS_VW_net_20bps,1.914657,20,156,0.046936,0.138627,0.603323,0.480219,1.172860,461.670423,-0.257452,-0.162144
9,ENET_LS_VW_net_30bps,1.914657,30,156,0.045021,0.138531,0.568405,0.479887,1.125793,346.501321,-0.260879,-0.164752


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,ENET_LS_EW_net_30bps,benchmark_equal_excess,156,0.031094,0.330381,-0.099003
1,value,ENET_LS_VW_net_30bps,benchmark_value_excess,156,0.037360,0.246640,-0.030504


,Feature,R2OOS,red_R2OOS,var_imp
0,cur_assets_assets,0.015024,2.340732e-03,1.420756e+00
1,ebitda_assets_raw,0.015914,1.451091e-03,8.807699e-01
2,Bid_Ask,0.016391,9.738124e-04,5.910760e-01
3,be_me,0.016439,9.260239e-04,5.620697e-01
4,ret_12_1_x_vn_mkt_chg1m,0.016521,8.438318e-04,5.121815e-01
5,oc_ret_1m,0.017167,1.982115e-04,1.203087e-01
6,ebitda_growth_12m_raw,0.017225,1.399565e-04,8.494954e-02
7,mom36m_x_idiovol,0.017233,1.323940e-04,8.035934e-02
8,cur_liab_assets,0.017284,8.167851e-05,4.957649e-02
9,cash_ratio,0.017365,4.407069e-08,2.674963e-05


,window_id,test_start,test_end,best_score,best_alpha,best_l1_ratio,best_max_iter,n_nonzero_coef
0,1,2012-03-31,2013-02-28,0.126502,0.00400,0.5,10000,14
1,2,2013-03-31,2014-02-28,0.125379,0.00232,0.5,10000,20
2,3,2014-03-31,2015-02-28,0.126826,0.00400,0.5,10000,12
3,4,2015-03-31,2016-02-29,0.113399,0.00400,0.5,10000,8
4,5,2016-03-31,2017-02-28,0.107476,0.00400,0.5,10000,10
5,6,2017-03-31,2018-02-28,0.133751,0.00358,0.5,10000,10
6,7,2018-03-31,2019-02-28,0.143123,0.00400,0.5,10000,11
7,8,2019-03-31,2020-02-29,0.130169,0.00400,0.5,10000,9
8,9,2020-03-31,2021-02-28,0.126033,0.00001,0.5,10000,132
9,10,2021-03-31,2022-02-28,0.152434,0.00127,0.5,10000,32


In [29]:
show_latest_recommendations('ENET', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,0.037043,1.890000e+03,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,FDC VM Equity,2026-02-28,0.032804,7.145345e+05,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
2,3,SMC VM Equity,2026-02-28,0.032315,1.215000e+04,8.943231e+05,4.008225e+09,10546.411312,-0.122744,0.029661,1.018272,0.750421
3,4,BCG VM Equity,2026-02-28,0.031228,2.530000e+03,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
4,5,ASP VM Equity,2026-02-28,0.030991,5.020000e+03,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117
5,6,DXP VH Equity,2026-02-28,0.030642,7.728407e+05,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881
6,7,DTD VH Equity,2026-02-28,0.027613,1.180343e+06,6.557330e+05,1.106520e+11,5.620928,0.000000,-0.123762,-0.103419,2.025794
7,8,AGG VM Equity,2026-02-28,0.027323,1.440000e+04,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
8,9,FMC VM Equity,2026-02-28,0.027237,2.811722e+06,1.909705e+06,4.540653e+11,4.464135,0.131579,0.162162,-0.084132,1.230356
9,10,AAT VM Equity,2026-02-28,0.026668,3.140000e+03,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T203049Z/recommendations/enet_max_v3_latest_recommendations.csv


## PLS

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [30]:
RUN_DIR_PLS_CAREFUL = run_single_model('PLS', feature_profile='careful_v3')
RUN_DIR_PLS_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models PLS --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWar

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T204337Z')

In [31]:
show_model_summary('PLS', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T204337Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_PLS,n_rows,n_months
0,Full sample,-0.008098,12716,156
1,Large firms,-0.026578,3756,156
2,Small firms,-0.010608,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,PLS_LS_EW_gross,1.409555,0,156,0.032589,0.066437,0.433481,0.230143,1.699249,106.918234,-0.175690,-0.151729
1,PLS_LS_EW_net_0bps,1.409555,0,156,0.032589,0.066437,0.433481,0.230143,1.699249,106.918234,-0.175690,-0.151729
2,PLS_LS_EW_net_10bps,1.409555,10,156,0.031180,0.066290,0.410236,0.229637,1.629335,86.255271,-0.179614,-0.152093
3,PLS_LS_EW_net_20bps,1.409555,20,156,0.029770,0.066152,0.387328,0.229158,1.558925,69.522726,-0.183527,-0.152456
4,PLS_LS_EW_net_30bps,1.409555,30,156,0.028361,0.066022,0.364754,0.228708,1.488040,55.977900,-0.188633,-0.152820
5,PLS_LS_VW_gross,1.754017,0,156,0.025397,0.092960,0.286514,0.322024,0.946386,25.447730,-0.365989,-0.214232
6,PLS_LS_VW_net_0bps,1.754017,0,156,0.025397,0.092960,0.286514,0.322024,0.946386,25.447730,-0.365989,-0.214232
7,PLS_LS_VW_net_10bps,1.754017,10,156,0.023643,0.092928,0.260179,0.321912,0.881329,19.212382,-0.383899,-0.215927
8,PLS_LS_VW_net_20bps,1.754017,20,156,0.021889,0.092906,0.234324,0.321837,0.816135,14.437550,-0.428295,-0.217622
9,PLS_LS_VW_net_30bps,1.754017,30,156,0.020135,0.092895,0.208942,0.321798,0.750825,10.783373,-0.469588,-0.219317


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,PLS_LS_EW_net_30bps,benchmark_equal_excess,156,0.020700,0.247086,0.080984
1,value,PLS_LS_VW_net_30bps,benchmark_value_excess,156,0.012474,0.118991,0.085771


,Feature,R2OOS,red_R2OOS,var_imp
0,cfp,0.019770,-0.001991,0.252027
1,pchsale_pchinvt,0.019745,-0.001966,0.248857
2,age,0.019699,-0.001920,0.243020
3,chinv,0.019694,-0.001915,0.242487
4,Free_Float_Pct,0.019650,-0.001871,0.236916
5,roeq,0.019646,-0.001867,0.236396
6,ep,0.019635,-0.001856,0.234960
7,maxret,0.019634,-0.001855,0.234874
8,be_me,0.018789,-0.001010,0.127870
9,std_turn,0.018192,-0.000413,0.052308


,window_id,test_start,test_end,best_score,best_n_components,n_components
0,1,2012-03-31,2013-02-28,0.125979,1,1
1,2,2013-03-31,2014-02-28,0.126001,2,2
2,3,2014-03-31,2015-02-28,0.127445,2,2
3,4,2015-03-31,2016-02-29,0.114305,1,1
4,5,2016-03-31,2017-02-28,0.107607,1,1
5,6,2017-03-31,2018-02-28,0.133746,1,1
6,7,2018-03-31,2019-02-28,0.143039,1,1
7,8,2019-03-31,2020-02-29,0.130474,1,1
8,9,2020-03-31,2021-02-28,0.126831,2,2
9,10,2021-03-31,2022-02-28,0.152182,7,7


In [32]:
show_latest_recommendations('PLS', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,0.044272,1.890000e+03,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,DVM VH Equity,2026-02-28,0.042882,3.011698e+05,6.628942e+05,7.570968e+09,4.488833,0.185185,-0.035619,-0.181397,1.170660
2,3,DXP VH Equity,2026-02-28,0.040439,7.728407e+05,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881
3,4,FMC VM Equity,2026-02-28,0.039974,2.811722e+06,1.909705e+06,4.540653e+11,4.464135,0.131579,0.162162,-0.084132,1.230356
4,5,TDC VM Equity,2026-02-28,0.039618,1.150000e+04,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
5,6,DTG VH Equity,2026-02-28,0.039487,1.460008e+05,1.446552e+05,2.403949e+09,1.090418,-0.044025,0.041096,-0.219654,1.374816
6,7,TD6 VH Equity,2026-02-28,0.039453,7.200000e+03,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
7,8,EVE VM Equity,2026-02-28,0.038994,4.512826e+05,3.008185e+05,1.893630e+10,4.231533,-0.004630,-0.004630,-0.031532,3.155350
8,9,FDC VM Equity,2026-02-28,0.038848,7.145345e+05,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
9,10,ASP VM Equity,2026-02-28,0.038565,5.020000e+03,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T204337Z/recommendations/pls_careful_v3_latest_recommendations.csv


### max_v3

In [33]:
RUN_DIR_PLS_MAX = run_single_model('PLS', feature_profile='max_v3')
RUN_DIR_PLS_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models PLS --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T204639Z')

In [34]:
show_model_summary('PLS', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T204639Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_PLS,n_rows,n_months
0,Full sample,-0.005440,12716,156
1,Large firms,-0.031324,3756,156
2,Small firms,-0.010004,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,PLS_LS_EW_gross,1.553461,0,156,0.041535,0.071267,0.585557,0.246876,2.018925,399.281313,-0.269060,-0.118620
1,PLS_LS_EW_net_0bps,1.553461,0,156,0.041535,0.071267,0.585557,0.246876,2.018925,399.281313,-0.269060,-0.118620
2,PLS_LS_EW_net_10bps,1.553461,10,156,0.039982,0.071069,0.557526,0.246189,1.948836,316.437611,-0.275826,-0.118906
3,PLS_LS_EW_net_20bps,1.553461,20,156,0.038428,0.070879,0.529939,0.245532,1.878132,250.630722,-0.282539,-0.119192
4,PLS_LS_EW_net_30bps,1.553461,30,156,0.036875,0.070698,0.502791,0.244904,1.806829,198.379428,-0.289200,-0.119477
5,PLS_LS_VW_gross,2.026602,0,156,0.052627,0.141248,0.708688,0.489298,1.290678,1057.318295,-0.282447,-0.213519
6,PLS_LS_VW_net_0bps,2.026602,0,156,0.052627,0.141248,0.708688,0.489298,1.290678,1057.318295,-0.282447,-0.213519
7,PLS_LS_VW_net_10bps,2.026602,10,156,0.050601,0.141133,0.669394,0.488900,1.241986,781.104970,-0.286276,-0.214609
8,PLS_LS_VW_net_20bps,2.026602,20,156,0.048574,0.141026,0.630909,0.488529,1.193148,576.545635,-0.290097,-0.215700
9,PLS_LS_VW_net_30bps,2.026602,30,156,0.046547,0.140927,0.593218,0.488186,1.144171,425.165437,-0.293910,-0.216790


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,PLS_LS_EW_net_30bps,benchmark_equal_excess,156,0.029214,0.304915,-0.112607
1,value,PLS_LS_VW_net_30bps,benchmark_value_excess,156,0.038887,0.250719,-0.057357


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.014376,0.006793,0.323805
1,ret_12_1_x_vn_mkt_chg1m,0.014553,0.006616,0.315375
2,mom36m_x_idiovol,0.014611,0.006558,0.312632
3,cur_assets_assets,0.014866,0.006303,0.300450
4,be_me,0.020467,0.000702,0.033465
5,ebitda_assets_raw,0.020628,0.000541,0.025777
6,Vol_90D,0.020788,0.000381,0.018142
7,oc_ret_1m,0.020894,0.000275,0.013112
8,ppe_growth_12m,0.020901,0.000268,0.012789
9,Vol_30D,0.020951,0.000218,0.010394


,window_id,test_start,test_end,best_score,best_n_components,n_components
0,1,2012-03-31,2013-02-28,0.125831,1,1
1,2,2013-03-31,2014-02-28,0.125941,2,2
2,3,2014-03-31,2015-02-28,0.127411,2,2
3,4,2015-03-31,2016-02-29,0.114277,1,1
4,5,2016-03-31,2017-02-28,0.107292,1,1
5,6,2017-03-31,2018-02-28,0.133361,1,1
6,7,2018-03-31,2019-02-28,0.142994,1,1
7,8,2019-03-31,2020-02-29,0.130588,1,1
8,9,2020-03-31,2021-02-28,0.127003,3,3
9,10,2021-03-31,2022-02-28,0.152582,3,3


In [35]:
show_latest_recommendations('PLS', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,0.057470,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,FDC VM Equity,2026-02-28,0.047406,714534.5280,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
2,3,AGG VM Equity,2026-02-28,0.046779,14400.0000,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
3,4,BCG VM Equity,2026-02-28,0.043492,2530.0000,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
4,5,TD6 VH Equity,2026-02-28,0.040208,7200.0000,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
5,6,ADS VM Equity,2026-02-28,0.039785,8400.0000,6.417157e+05,4.728175e+08,640.096509,0.021898,-0.015240,-0.133127,1.510058
6,7,API VH Equity,2026-02-28,0.038861,6300.0000,5.297290e+05,1.016526e+09,776.604533,-0.045455,-0.231707,-0.171053,1.702812
7,8,SMC VM Equity,2026-02-28,0.038473,12150.0000,8.943231e+05,4.008225e+09,10546.411312,-0.122744,0.029661,1.018272,0.750421
8,9,FIR VM Equity,2026-02-28,0.037356,438858.3402,6.322401e+05,1.531971e+09,0.509506,-0.133891,-0.267689,-0.014288,1.177755
9,10,BCC VH Equity,2026-02-28,0.036858,7800.0000,9.610365e+05,5.184796e+08,254.922883,-0.025000,-0.071429,-0.037037,1.918663


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T204639Z/recommendations/pls_max_v3_latest_recommendations.csv


## PCR

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [36]:
RUN_DIR_PCR_CAREFUL = run_single_model('PCR', feature_profile='careful_v3')
RUN_DIR_PCR_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models PCR --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWar

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T205458Z')

In [37]:
show_model_summary('PCR', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T205458Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_PCR,n_rows,n_months
0,Full sample,-0.011471,12716,156
1,Large firms,-0.016684,3756,156
2,Small firms,-0.004008,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,PCR_LS_EW_gross,1.430729,0,156,0.036941,0.065116,0.509801,0.225569,1.965208,210.813994,-0.164092,-0.116134
1,PCR_LS_EW_net_0bps,1.430729,0,156,0.036941,0.065116,0.509801,0.225569,1.965208,210.813994,-0.164092,-0.116134
2,PCR_LS_EW_net_10bps,1.430729,10,156,0.035510,0.064961,0.485059,0.225030,1.893620,169.871420,-0.172318,-0.116420
3,PCR_LS_EW_net_20bps,1.430729,20,156,0.034079,0.064813,0.460682,0.224517,1.821478,136.792164,-0.180477,-0.116706
4,PCR_LS_EW_net_30bps,1.430729,30,156,0.032649,0.064672,0.436664,0.224030,1.748800,110.075756,-0.188569,-0.116992
5,PCR_LS_VW_gross,1.798990,0,156,0.023668,0.084437,0.270019,0.292498,0.971017,21.363052,-0.460408,-0.276455
6,PCR_LS_VW_net_0bps,1.798990,0,156,0.023668,0.084437,0.270019,0.292498,0.971017,21.363052,-0.460408,-0.276455
7,PCR_LS_VW_net_10bps,1.798990,10,156,0.021869,0.084377,0.243368,0.292290,0.897852,15.974497,-0.490145,-0.278478
8,PCR_LS_VW_net_20bps,1.798990,20,156,0.020070,0.084329,0.217216,0.292124,0.824462,11.875981,-0.518316,-0.280500
9,PCR_LS_VW_net_30bps,1.798990,30,156,0.018271,0.084293,0.191554,0.292000,0.750879,8.760673,-0.544999,-0.282522


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,PCR_LS_EW_net_30bps,benchmark_equal_excess,156,0.024988,0.285319,-0.029283
1,value,PCR_LS_VW_net_30bps,benchmark_value_excess,156,0.010611,0.103103,-0.022267


,Feature,R2OOS,red_R2OOS,var_imp
0,ep,0.005152,8.381766e-03,1.618558
1,mom6m,0.005506,8.027581e-03,1.550163
2,tri_ret_12_1,0.005581,7.953090e-03,1.535779
3,mom36m,0.005642,7.891565e-03,1.523898
4,FCF,0.007246,6.287866e-03,1.214216
5,be_me,0.012520,1.013378e-03,0.195688
6,lev,0.012533,1.000642e-03,0.193229
7,Bid_Ask,0.012782,7.515381e-04,0.145126
8,cash_ratio,0.012960,5.734364e-04,0.110733
9,chcsho,0.013397,1.370427e-04,0.026464


,window_id,test_start,test_end,best_score,best_n_components,n_components
0,1,2012-03-31,2013-02-28,0.125895,7,7
1,2,2013-03-31,2014-02-28,0.126027,11,11
2,3,2014-03-31,2015-02-28,0.127266,22,22
3,4,2015-03-31,2016-02-29,0.113925,3,3
4,5,2016-03-31,2017-02-28,0.107602,3,3
5,6,2017-03-31,2018-02-28,0.133519,11,11
6,7,2018-03-31,2019-02-28,0.142880,15,15
7,8,2019-03-31,2020-02-29,0.129943,49,49
8,9,2020-03-31,2021-02-28,0.124530,33,33
9,10,2021-03-31,2022-02-28,0.152254,25,25


In [38]:
show_latest_recommendations('PCR', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TD6 VH Equity,2026-02-28,0.041009,7.200000e+03,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
1,2,TDC VM Equity,2026-02-28,0.035295,1.150000e+04,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
2,3,TCD VM Equity,2026-02-28,0.034599,1.890000e+03,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
3,4,ASP VM Equity,2026-02-28,0.034569,5.020000e+03,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117
4,5,AAT VM Equity,2026-02-28,0.034517,3.140000e+03,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
5,6,FDC VM Equity,2026-02-28,0.032611,7.145345e+05,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
6,7,EVE VM Equity,2026-02-28,0.031792,4.512826e+05,3.008185e+05,1.893630e+10,4.231533,-0.004630,-0.004630,-0.031532,3.155350
7,8,DTG VH Equity,2026-02-28,0.031551,1.460008e+05,1.446552e+05,2.403949e+09,1.090418,-0.044025,0.041096,-0.219654,1.374816
8,9,FIR VM Equity,2026-02-28,0.030605,4.388583e+05,6.322401e+05,1.531971e+09,0.509506,-0.133891,-0.267689,-0.014288,1.177755
9,10,FMC VM Equity,2026-02-28,0.030132,2.811722e+06,1.909705e+06,4.540653e+11,4.464135,0.131579,0.162162,-0.084132,1.230356


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T205458Z/recommendations/pcr_careful_v3_latest_recommendations.csv


### max_v3

In [39]:
RUN_DIR_PCR_MAX = run_single_model('PCR', feature_profile='max_v3')
RUN_DIR_PCR_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models PCR --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T205907Z')

In [40]:
show_model_summary('PCR', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T205907Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_PCR,n_rows,n_months
0,Full sample,-0.002993,12716,156
1,Large firms,-0.016219,3756,156
2,Small firms,-0.005925,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,PCR_LS_EW_gross,1.466789,0,156,0.039335,0.065530,0.551319,0.227004,2.079350,300.378785,-0.230310,-0.186102
1,PCR_LS_EW_net_0bps,1.466789,0,156,0.039335,0.065530,0.551319,0.227004,2.079350,300.378785,-0.230310,-0.186102
2,PCR_LS_EW_net_10bps,1.466789,10,156,0.037868,0.065408,0.525282,0.226581,2.005547,240.854106,-0.237232,-0.187502
3,PCR_LS_EW_net_20bps,1.466789,20,156,0.036401,0.065299,0.499634,0.226202,1.931094,193.002436,-0.249692,-0.188902
4,PCR_LS_EW_net_30bps,1.466789,30,156,0.034935,0.065202,0.474367,0.225867,1.856029,154.551052,-0.261974,-0.190302
5,PCR_LS_VW_gross,1.835767,0,156,0.031363,0.089802,0.383038,0.311083,1.209806,66.739794,-0.593427,-0.286848
6,PCR_LS_VW_net_0bps,1.835767,0,156,0.031363,0.089802,0.383038,0.311083,1.209806,66.739794,-0.593427,-0.286848
7,PCR_LS_VW_net_10bps,1.835767,10,156,0.029527,0.089757,0.353599,0.310927,1.139565,50.211618,-0.603224,-0.289592
8,PCR_LS_VW_net_20bps,1.835767,20,156,0.027691,0.089726,0.324715,0.310818,1.069086,37.689302,-0.612810,-0.292336
9,PCR_LS_VW_net_30bps,1.835767,30,156,0.025855,0.089708,0.296378,0.310758,0.998404,28.208504,-0.622189,-0.295080


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,PCR_LS_EW_net_30bps,benchmark_equal_excess,156,0.027274,0.310343,-0.026972
1,value,PCR_LS_VW_net_30bps,benchmark_value_excess,156,0.018194,0.175855,0.059836


,Feature,R2OOS,red_R2OOS,var_imp
0,capex_assets_raw,0.023496,-0.008195,0.540063
1,idiovol,0.023311,-0.008010,0.527870
2,pchsale_pchinvt,0.023271,-0.007971,0.525278
3,agr,0.015803,-0.000502,0.033093
4,assets_growth_12m,0.015767,-0.000466,0.030693
5,gross_profit_growth_12m_raw,0.015675,-0.000374,0.024639
6,debt_growth_12m,0.015630,-0.000329,0.021687
7,mom6m_x_turn,0.015623,-0.000322,0.021236
8,adv_med,0.015601,-0.000300,0.019770
9,dollar_vol,0.015592,-0.000291,0.019187


,window_id,test_start,test_end,best_score,best_n_components,n_components
0,1,2012-03-31,2013-02-28,0.125954,7,7
1,2,2013-03-31,2014-02-28,0.125821,11,11
2,3,2014-03-31,2015-02-28,0.127420,11,11
3,4,2015-03-31,2016-02-29,0.113980,1,1
4,5,2016-03-31,2017-02-28,0.106954,9,9
5,6,2017-03-31,2018-02-28,0.133011,9,9
6,7,2018-03-31,2019-02-28,0.142672,11,11
7,8,2019-03-31,2020-02-29,0.130400,9,9
8,9,2020-03-31,2021-02-28,0.126788,15,15
9,10,2021-03-31,2022-02-28,0.152410,25,25


In [41]:
show_latest_recommendations('PCR', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,FDC VM Equity,2026-02-28,0.050754,714534.5280,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
1,2,TCD VM Equity,2026-02-28,0.042857,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
2,3,AGG VM Equity,2026-02-28,0.038627,14400.0000,2.340404e+06,5.218920e+09,2293.142207,-0.049505,-0.263427,-0.137725,1.425210
3,4,BCG VM Equity,2026-02-28,0.036578,2530.0000,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
4,5,TD6 VH Equity,2026-02-28,0.035798,7200.0000,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
5,6,API VH Equity,2026-02-28,0.033351,6300.0000,5.297290e+05,1.016526e+09,776.604533,-0.045455,-0.231707,-0.171053,1.702812
6,7,SMC VM Equity,2026-02-28,0.033017,12150.0000,8.943231e+05,4.008225e+09,10546.411312,-0.122744,0.029661,1.018272,0.750421
7,8,DTL VM Equity,2026-02-28,0.030144,685130.1192,1.819413e+06,6.738738e+09,0.768654,-0.143939,-0.017391,0.153061,0.350817
8,9,BCM VM Equity,2026-02-28,0.030066,67300.0000,6.965550e+07,5.219632e+10,1167.439614,-0.096644,0.012030,-0.101469,0.326085
9,10,ADS VM Equity,2026-02-28,0.029497,8400.0000,6.417157e+05,4.728175e+08,640.096509,0.021898,-0.015240,-0.133127,1.510058


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T205907Z/recommendations/pcr_max_v3_latest_recommendations.csv


## GBRT

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [42]:
RUN_DIR_GBRT_CAREFUL = run_single_model('GBRT', feature_profile='careful_v3')
RUN_DIR_GBRT_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models GBRT --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWa

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T210902Z')

In [43]:
show_model_summary('GBRT', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T210902Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_GBRT,n_rows,n_months
0,Full sample,-0.072506,12716,156
1,Large firms,-0.011899,3756,156
2,Small firms,-0.003637,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,GBRT_LS_EW_gross,1.480556,0,36,0.038395,0.068147,0.533561,0.236068,1.951721,2.606642,-0.103399,-0.060875
1,GBRT_LS_EW_net_0bps,1.480556,0,36,0.038395,0.068147,0.533561,0.236068,1.951721,2.606642,-0.103399,-0.060875
2,GBRT_LS_EW_net_10bps,1.480556,10,36,0.036914,0.067915,0.507684,0.235263,1.882879,2.427132,-0.109359,-0.062147
3,GBRT_LS_EW_net_20bps,1.480556,20,36,0.035434,0.067691,0.482198,0.234490,1.813325,2.256255,-0.115292,-0.063420
4,GBRT_LS_EW_net_30bps,1.480556,30,36,0.033953,0.067477,0.457097,0.233747,1.743079,2.093609,-0.121199,-0.064693
5,GBRT_LS_VW_gross,1.805647,0,36,0.066834,0.275442,0.721315,0.954158,0.840540,4.100130,-0.310201,-0.207427
6,GBRT_LS_VW_net_0bps,1.805647,0,36,0.066834,0.275442,0.721315,0.954158,0.840540,4.100130,-0.310201,-0.207427
7,GBRT_LS_VW_net_10bps,1.805647,10,36,0.065028,0.275481,0.685270,0.954295,0.817714,3.786393,-0.315455,-0.210347
8,GBRT_LS_VW_net_20bps,1.805647,20,36,0.063223,0.275523,0.649902,0.954440,0.794888,3.491325,-0.320684,-0.213267
9,GBRT_LS_VW_net_30bps,1.805647,30,36,0.061417,0.275567,0.615201,0.954594,0.772061,3.213853,-0.325890,-0.216187


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,GBRT_LS_EW_net_30bps,benchmark_equal_excess,36,0.040156,0.526675,0.193229
1,value,GBRT_LS_VW_net_30bps,benchmark_value_excess,36,0.067619,0.249176,0.174133


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.001303,0.0,NaN
1,China_Shanghai_Index,0.001303,0.0,NaN
2,Comm_Brent_Oil,0.001303,0.0,NaN
3,Comm_Copper,0.001303,0.0,NaN
4,Comm_Gold_Spot,0.001303,0.0,NaN
5,Comm_Natural_Gas,0.001303,0.0,NaN
6,FCF,0.001303,0.0,NaN
7,Free_Float_Pct,0.001303,0.0,NaN
8,Global_Baltic_Dry,0.001303,0.0,NaN
9,Hong_Kong_Index,0.001303,0.0,NaN


,window_id,test_start,test_end,best_score,best_learning_rate,best_max_depth,best_max_features,best_min_samples_leaf,best_min_samples_split,best_n_estimators
0,1,2012-03-31,2013-02-28,8.645137,0.01,1,sqrt,50,5000,100
1,2,2013-03-31,2014-02-28,9.557177,0.10,1,sqrt,50,5000,100
2,3,2014-03-31,2015-02-28,10.306700,0.10,1,sqrt,50,5000,100
3,4,2015-03-31,2016-02-29,8.574912,0.10,1,sqrt,50,5000,100
4,5,2016-03-31,2017-02-28,8.379037,0.10,1,sqrt,50,5000,100
5,6,2017-03-31,2018-02-28,14.097843,0.10,1,sqrt,50,5000,100
6,7,2018-03-31,2019-02-28,17.503444,0.10,1,sqrt,50,5000,100
7,8,2019-03-31,2020-02-29,16.025851,0.01,1,sqrt,50,5000,100
8,9,2020-03-31,2021-02-28,15.860605,0.01,1,sqrt,50,5000,100
9,10,2021-03-31,2022-02-28,25.070407,0.10,1,sqrt,50,5000,100


In [44]:
show_latest_recommendations('GBRT', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,AAA VM Equity,2026-02-28,0.001466,7900.0,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
1,2,AAT VM Equity,2026-02-28,0.001466,3140.0,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
2,3,AAV VH Equity,2026-02-28,0.001466,6100.0,4.208247e+05,1.110685e+09,12306.679195,0.033898,0.033898,-0.140845,1.674249
3,4,ABS VM Equity,2026-02-28,0.001466,2880.0,2.304000e+05,4.351015e+08,697.500000,-0.043189,-0.234043,-0.391121,1.758511
4,5,ABT VM Equity,2026-02-28,0.001466,66500.0,7.831876e+05,4.471000e+08,1154.767747,-0.056738,0.108333,0.504525,0.854800
5,6,ACB VM Equity,2026-02-28,0.001466,24550.0,1.261049e+08,3.049721e+11,2151.399414,-0.012072,-0.118492,0.077576,0.749532
6,7,ACC VM Equity,2026-02-28,0.001466,12700.0,1.333500e+06,1.953600e+08,54.285719,-0.034221,-0.086331,-0.124138,1.033665
7,8,ACG VM Equity,2026-02-28,0.001466,36250.0,5.466063e+06,7.079625e+08,704.963493,0.000000,-0.032043,-0.065722,0.802317
8,9,ACL VM Equity,2026-02-28,0.001466,13800.0,6.921945e+05,1.792700e+08,93.701990,-0.003610,0.126531,0.174468,1.225571
9,10,ADP VM Equity,2026-02-28,0.001466,23800.0,5.483484e+05,1.508875e+08,598.962233,-0.008333,-0.108614,-0.173611,0.483061


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T210902Z/recommendations/gbrt_careful_v3_latest_recommendations.csv


### max_v3

In [46]:
RUN_DIR_GBRT_MAX = run_single_model('GBRT', feature_profile='max_v3')
RUN_DIR_GBRT_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models GBRT --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarnin

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260310T231503Z')

In [47]:
show_model_summary('GBRT', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260310T231503Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_GBRT,n_rows,n_months
0,Full sample,-0.071740,12716,156
1,Large firms,-0.011899,3756,156
2,Small firms,-0.003637,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,GBRT_LS_EW_gross,1.446184,0,36,0.031890,0.077983,0.410852,0.270142,1.416594,1.808305,-0.207440,-0.114186
1,GBRT_LS_EW_net_0bps,1.446184,0,36,0.031890,0.077983,0.410852,0.270142,1.416594,1.808305,-0.207440,-0.114186
2,GBRT_LS_EW_net_10bps,1.446184,10,36,0.030444,0.077785,0.387434,0.269456,1.355799,1.670772,-0.210641,-0.114408
3,GBRT_LS_EW_net_20bps,1.446184,20,36,0.028998,0.077600,0.364358,0.268813,1.294482,1.539713,-0.213835,-0.114630
4,GBRT_LS_EW_net_30bps,1.446184,30,36,0.027552,0.077427,0.341619,0.268214,1.232668,1.414834,-0.217023,-0.114852
5,GBRT_LS_VW_gross,1.936676,0,36,0.065642,0.254909,0.744915,0.883031,0.892049,4.312792,-0.164632,-0.140468
6,GBRT_LS_VW_net_0bps,1.936676,0,36,0.065642,0.254909,0.744915,0.883031,0.892049,4.312792,-0.164632,-0.140468
7,GBRT_LS_VW_net_10bps,1.936676,10,36,0.063706,0.254749,0.706475,0.882476,0.866275,3.969355,-0.182785,-0.143104
8,GBRT_LS_VW_net_20bps,1.936676,20,36,0.061769,0.254593,0.668790,0.881937,0.840453,3.647350,-0.202552,-0.145739
9,GBRT_LS_VW_net_30bps,1.936676,30,36,0.059832,0.254442,0.631847,0.881414,0.814585,3.345487,-0.221888,-0.148375


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,GBRT_LS_EW_net_30bps,benchmark_equal_excess,36,0.033754,0.380960,0.092776
1,value,GBRT_LS_VW_net_30bps,benchmark_value_excess,36,0.066035,0.264203,0.187691


,Feature,R2OOS,red_R2OOS,var_imp
0,Bid_Ask,0.001303,0.0,NaN
1,China_Shanghai_Index,0.001303,0.0,NaN
2,China_Shanghai_Index_chg1m,0.001303,0.0,NaN
3,China_Shanghai_Index_chg3m,0.001303,0.0,NaN
4,Comm_Brent_Oil,0.001303,0.0,NaN
5,Comm_Brent_Oil_chg1m,0.001303,0.0,NaN
6,Comm_Brent_Oil_chg3m,0.001303,0.0,NaN
7,Comm_Copper,0.001303,0.0,NaN
8,Comm_Copper_chg1m,0.001303,0.0,NaN
9,Comm_Copper_chg3m,0.001303,0.0,NaN


,window_id,test_start,test_end,best_score,best_learning_rate,best_max_depth,best_max_features,best_min_samples_leaf,best_min_samples_split,best_n_estimators
0,1,2012-03-31,2013-02-28,8.645137,0.01,1,sqrt,50,5000,100
1,2,2013-03-31,2014-02-28,9.557177,0.10,1,sqrt,50,5000,100
2,3,2014-03-31,2015-02-28,10.306700,0.10,1,sqrt,50,5000,100
3,4,2015-03-31,2016-02-29,8.574912,0.10,1,sqrt,50,5000,100
4,5,2016-03-31,2017-02-28,8.379037,0.10,1,sqrt,50,5000,100
5,6,2017-03-31,2018-02-28,14.097843,0.10,1,sqrt,50,5000,100
6,7,2018-03-31,2019-02-28,17.503444,0.10,1,sqrt,50,5000,100
7,8,2019-03-31,2020-02-29,16.025851,0.01,1,sqrt,50,5000,100
8,9,2020-03-31,2021-02-28,15.860605,0.01,1,sqrt,50,5000,100
9,10,2021-03-31,2022-02-28,25.070407,0.10,1,sqrt,50,5000,100


In [48]:
show_latest_recommendations('GBRT', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,AAA VM Equity,2026-02-28,0.001466,7900.0,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
1,2,AAT VM Equity,2026-02-28,0.001466,3140.0,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
2,3,AAV VH Equity,2026-02-28,0.001466,6100.0,4.208247e+05,1.110685e+09,12306.679195,0.033898,0.033898,-0.140845,1.674249
3,4,ABS VM Equity,2026-02-28,0.001466,2880.0,2.304000e+05,4.351015e+08,697.500000,-0.043189,-0.234043,-0.391121,1.758511
4,5,ABT VM Equity,2026-02-28,0.001466,66500.0,7.831876e+05,4.471000e+08,1154.767747,-0.056738,0.108333,0.504525,0.854800
5,6,ACB VM Equity,2026-02-28,0.001466,24550.0,1.261049e+08,3.049721e+11,2151.399414,-0.012072,-0.118492,0.077576,0.749532
6,7,ACC VM Equity,2026-02-28,0.001466,12700.0,1.333500e+06,1.953600e+08,54.285719,-0.034221,-0.086331,-0.124138,1.033665
7,8,ACG VM Equity,2026-02-28,0.001466,36250.0,5.466063e+06,7.079625e+08,704.963493,0.000000,-0.032043,-0.065722,0.802317
8,9,ACL VM Equity,2026-02-28,0.001466,13800.0,6.921945e+05,1.792700e+08,93.701990,-0.003610,0.126531,0.174468,1.225571
9,10,ADP VM Equity,2026-02-28,0.001466,23800.0,5.483484e+05,1.508875e+08,598.962233,-0.008333,-0.108614,-0.173611,0.483061


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260310T231503Z/recommendations/gbrt_max_v3_latest_recommendations.csv


## RF

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [49]:
RUN_DIR_RF_CAREFUL = run_single_model('RF', feature_profile='careful_v3')
RUN_DIR_RF_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models RF --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarn

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260311T010357Z')

In [50]:
show_model_summary('RF', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260311T010357Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_RF,n_rows,n_months
0,Full sample,-0.162650,12716,156
1,Large firms,-0.190125,3756,156
2,Small firms,-0.064156,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,RF_LS_EW_gross,1.730620,0,132,0.027706,0.076626,0.342614,0.265440,1.252511,24.554214,-0.395578,-0.140186
1,RF_LS_EW_net_0bps,1.730620,0,132,0.027706,0.076626,0.342614,0.265440,1.252511,24.554214,-0.395578,-0.140186
2,RF_LS_EW_net_10bps,1.730620,10,132,0.025975,0.076491,0.315742,0.264973,1.176343,19.458670,-0.405438,-0.141519
3,RF_LS_EW_net_20bps,1.730620,20,132,0.024244,0.076366,0.289352,0.264540,1.099761,15.371492,-0.415160,-0.142852
4,RF_LS_EW_net_30bps,1.730620,30,132,0.022514,0.076252,0.263438,0.264143,1.022793,12.094656,-0.424745,-0.144186
5,RF_LS_VW_gross,1.946176,0,132,0.028846,0.081659,0.354410,0.282874,1.223686,27.135401,-0.260827,-0.154504
6,RF_LS_VW_net_0bps,1.946176,0,132,0.028846,0.081659,0.354410,0.282874,1.223686,27.135401,-0.260827,-0.154504
7,RF_LS_VW_net_10bps,1.946176,10,132,0.026900,0.081539,0.323939,0.282459,1.142806,20.905174,-0.265934,-0.158219
8,RF_LS_VW_net_20bps,1.946176,20,132,0.024953,0.081432,0.294080,0.282090,1.061510,16.044081,-0.271021,-0.161934
9,RF_LS_VW_net_30bps,1.946176,30,132,0.023007,0.081339,0.264824,0.281768,0.979838,12.253550,-0.276087,-0.165649


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,RF_LS_EW_net_30bps,benchmark_equal_excess,132,0.018090,0.203000,0.062253
1,value,RF_LS_VW_net_30bps,benchmark_value_excess,132,0.018583,0.189021,-0.052645


,Feature,R2OOS,red_R2OOS,var_imp
0,VN_MoneySupply_M2,0.028780,0.000979,0.147601
1,Textile_Cotton_Price,0.028934,0.000825,0.124313
2,US_GDP_QoQ,0.029008,0.000751,0.113151
3,Hong_Kong_Index,0.029058,0.000701,0.105690
4,Thailand_Index,0.029339,0.000420,0.063247
5,ep,0.029386,0.000373,0.056229
6,VN_DIAMOND_INDEX,0.029393,0.000366,0.055138
7,Comm_Copper,0.029441,0.000318,0.047883
8,FCF,0.029442,0.000317,0.047803
9,US_CPI_YoY,0.029461,0.000298,0.044880


,window_id,test_start,test_end,best_score,best_max_depth,best_max_features,best_n_estimators
0,1,2012-03-31,2013-02-28,0.015693,3,12,100
1,2,2013-03-31,2014-02-28,0.015942,2,49,100
2,3,2014-03-31,2015-02-28,0.016877,1,49,100
3,4,2015-03-31,2016-02-29,0.013144,4,24,100
4,5,2016-03-31,2017-02-28,0.011591,6,6,100
5,6,2017-03-31,2018-02-28,0.017760,1,46,100
6,7,2018-03-31,2019-02-28,0.020714,1,3,100
7,8,2019-03-31,2020-02-29,0.016982,1,3,100
8,9,2020-03-31,2021-02-28,0.015729,1,6,100
9,10,2021-03-31,2022-02-28,0.023391,1,46,100


In [51]:
show_latest_recommendations('RF', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,TCD VM Equity,2026-02-28,-0.009809,1.890000e+03,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
1,2,DQC VM Equity,2026-02-28,-0.009909,2.824702e+05,3.252736e+05,1.844365e+09,0.643277,0.004902,-0.123932,-0.088889,2.384791
2,3,DVM VH Equity,2026-02-28,-0.010326,3.011698e+05,6.628942e+05,7.570968e+09,4.488833,0.185185,-0.035619,-0.181397,1.170660
3,4,FCN VM Equity,2026-02-28,-0.010519,2.141170e+06,6.365212e+06,4.649552e+11,15.535603,-0.071672,-0.235955,-0.160494,0.397581
4,5,FMC VM Equity,2026-02-28,-0.010652,2.811722e+06,1.909705e+06,4.540653e+11,4.464135,0.131579,0.162162,-0.084132,1.230356
5,6,SZC VM Equity,2026-02-28,-0.010675,3.450000e+04,6.209512e+06,2.089451e+10,25002.073632,0.081505,-0.008621,-0.215017,0.515854
6,7,DXP VH Equity,2026-02-28,-0.010696,7.728407e+05,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881
7,8,ASP VM Equity,2026-02-28,-0.010789,5.020000e+03,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117
8,9,TD6 VH Equity,2026-02-28,-0.010789,7.200000e+03,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
9,10,EVS VH Equity,2026-02-28,-0.010861,9.228835e+05,1.144406e+06,1.507224e+10,3.136401,-0.050847,-0.291139,-0.125000,1.729100


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260311T010357Z/recommendations/rf_careful_v3_latest_recommendations.csv


### max_v3

In [52]:
RUN_DIR_RF_MAX = run_single_model('RF', feature_profile='max_v3')
RUN_DIR_RF_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models RF --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning:

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260311T012604Z')

In [53]:
show_model_summary('RF', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260311T012604Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_RF,n_rows,n_months
0,Full sample,-0.109068,12716,156
1,Large firms,-0.199395,3756,156
2,Small firms,-0.047990,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,RF_LS_EW_gross,1.771114,0,144,0.012950,0.073224,0.130921,0.253657,0.612650,3.377102,-0.614612,-0.202348
1,RF_LS_EW_net_0bps,1.771114,0,144,0.012950,0.073224,0.130921,0.253657,0.612650,3.377102,-0.614612,-0.202348
2,RF_LS_EW_net_10bps,1.771114,10,144,0.011179,0.073130,0.107391,0.253329,0.529545,2.401026,-0.645908,-0.203014
3,RF_LS_EW_net_20bps,1.771114,20,144,0.009408,0.073047,0.084298,0.253044,0.446152,1.641108,-0.674726,-0.204074
4,RF_LS_EW_net_30bps,1.771114,30,144,0.007637,0.072977,0.061637,0.252800,0.362510,1.049810,-0.701256,-0.205217
5,RF_LS_VW_gross,2.113557,0,144,0.003040,0.098442,-0.022354,0.341012,0.106967,-0.237604,-0.856442,-0.281913
6,RF_LS_VW_net_0bps,2.113557,0,144,0.003040,0.098442,-0.022354,0.341012,0.106967,-0.237604,-0.856442,-0.281913
7,RF_LS_VW_net_10bps,2.113557,10,144,0.000926,0.098558,-0.047154,0.341415,0.032554,-0.439888,-0.878630,-0.284572
8,RF_LS_VW_net_20bps,2.113557,20,144,-0.001187,0.098687,-0.071391,0.341862,-0.041679,-0.588855,-0.898959,-0.287230
9,RF_LS_VW_net_30bps,2.113557,30,144,-0.003301,0.098829,-0.095077,0.342352,-0.115702,-0.698463,-0.919310,-0.289889


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,RF_LS_EW_net_30bps,benchmark_equal_excess,144,-0.000691,-0.007182,-0.052714
1,value,RF_LS_VW_net_30bps,benchmark_value_excess,144,-0.011629,-0.097478,-0.084299


,Feature,R2OOS,red_R2OOS,var_imp
0,dy,0.026651,0.005605,0.034246
1,Comm_Brent_Oil,0.026755,0.005501,0.033612
2,US_Market_SP500_chg1m,0.026861,0.005395,0.032963
3,Comm_Copper_chg3m,0.026942,0.005314,0.032467
4,Indonesia_Index_chg1m,0.026960,0.005296,0.032361
5,Comm_Gold_Spot,0.027021,0.005235,0.031984
6,US_GDP_QoQ,0.027027,0.005230,0.031953
7,Hong_Kong_Index_chg3m,0.027039,0.005217,0.031878
8,USD_CNY_FX,0.027044,0.005212,0.031846
9,VN_DIAMOND_INDEX_chg3m,0.027080,0.005176,0.031624


,window_id,test_start,test_end,best_score,best_max_depth,best_max_features,best_n_estimators
0,1,2012-03-31,2013-02-28,0.015815,3,6,100
1,2,2013-03-31,2014-02-28,0.015987,2,49,100
2,3,2014-03-31,2015-02-28,0.017068,1,49,100
3,4,2015-03-31,2016-02-29,0.013096,5,24,100
4,5,2016-03-31,2017-02-28,0.011573,6,6,100
5,6,2017-03-31,2018-02-28,0.017768,1,49,100
6,7,2018-03-31,2019-02-28,0.020660,1,3,100
7,8,2019-03-31,2020-02-29,0.016945,1,6,100
8,9,2020-03-31,2021-02-28,0.015767,1,3,100
9,10,2021-03-31,2022-02-28,0.023403,2,49,100


In [54]:
show_latest_recommendations('RF', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,AAA VM Equity,2026-02-28,-0.015193,7900.0,3.019969e+06,1.203270e+10,3461.125448,-0.025894,0.003812,-0.109357,1.810756
1,2,AAT VM Equity,2026-02-28,-0.015193,3140.0,2.223720e+05,8.812150e+07,303.590416,-0.057057,-0.068249,-0.059880,3.248986
2,3,AAV VH Equity,2026-02-28,-0.015193,6100.0,4.208247e+05,1.110685e+09,12306.679195,0.033898,0.033898,-0.140845,1.674249
3,4,ABS VM Equity,2026-02-28,-0.015193,2880.0,2.304000e+05,4.351015e+08,697.500000,-0.043189,-0.234043,-0.391121,1.758511
4,5,ABT VM Equity,2026-02-28,-0.015193,66500.0,7.831876e+05,4.471000e+08,1154.767747,-0.056738,0.108333,0.504525,0.854800
5,6,ACB VM Equity,2026-02-28,-0.015193,24550.0,1.261049e+08,3.049721e+11,2151.399414,-0.012072,-0.118492,0.077576,0.749532
6,7,ACC VM Equity,2026-02-28,-0.015193,12700.0,1.333500e+06,1.953600e+08,54.285719,-0.034221,-0.086331,-0.124138,1.033665
7,8,ACG VM Equity,2026-02-28,-0.015193,36250.0,5.466063e+06,7.079625e+08,704.963493,0.000000,-0.032043,-0.065722,0.802317
8,9,ACL VM Equity,2026-02-28,-0.015193,13800.0,6.921945e+05,1.792700e+08,93.701990,-0.003610,0.126531,0.174468,1.225571
9,10,ADP VM Equity,2026-02-28,-0.015193,23800.0,5.483484e+05,1.508875e+08,598.962233,-0.008333,-0.108614,-0.173611,0.483061


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260311T012604Z/recommendations/rf_max_v3_latest_recommendations.csv


## NN

This model is split into two comparable tracks. The first 3 blocks run, summarise, and recommend under `careful_v3`. The next 3 blocks do the same for `max_v3`.

### careful_v3

In [55]:
RUN_DIR_NN_CAREFUL = run_single_model('NN', feature_profile='careful_v3')
RUN_DIR_NN_CAREFUL

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/careful_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarn

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260311T021101Z')

In [56]:
show_model_summary('NN', feature_profile='careful_v3')

Feature profile: careful_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260311T021101Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,61
5,requested_optional_low_coverage,2
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,55
9,feature_profile,careful_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.006811,12716,156
1,Large firms,-0.062610,3756,156
2,Small firms,-0.044476,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,1.793351,0,156,0.039949,0.077183,0.549494,0.267368,1.793008,295.802600,-0.278066,-0.205329
1,NN_LS_EW_net_0bps,1.793351,0,156,0.039949,0.077183,0.549494,0.267368,1.793008,295.802600,-0.278066,-0.205329
2,NN_LS_EW_net_10bps,1.793351,10,156,0.038156,0.077030,0.517769,0.266840,1.715910,225.815642,-0.279598,-0.206579
3,NN_LS_EW_net_20bps,1.793351,20,156,0.036363,0.076883,0.486633,0.266331,1.638386,172.240331,-0.283646,-0.207829
4,NN_LS_EW_net_30bps,1.793351,30,156,0.034569,0.076742,0.456077,0.265842,1.560451,131.249703,-0.287851,-0.209079
5,NN_LS_VW_gross,2.064631,0,156,0.037743,0.102519,0.473533,0.355137,1.275321,153.411329,-0.441740,-0.229136
6,NN_LS_VW_net_0bps,2.064631,0,156,0.037743,0.102519,0.473533,0.355137,1.275321,153.411329,-0.441740,-0.229136
7,NN_LS_VW_net_10bps,2.064631,10,156,0.035678,0.102570,0.438309,0.355312,1.204966,111.739740,-0.452108,-0.232664
8,NN_LS_VW_net_20bps,2.064631,20,156,0.033614,0.102628,0.403843,0.355513,1.134595,81.250763,-0.462850,-0.236192
9,NN_LS_VW_net_30bps,2.064631,30,156,0.031549,0.102694,0.370121,0.355741,1.064223,58.960604,-0.473414,-0.239720


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.026909,0.288386,0.051716
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,0.023888,0.207046,0.043022


,Feature,R2OOS,red_R2OOS,var_imp
0,be_me,0.020996,0.003759,0.164749
1,Bid_Ask,0.022442,0.002314,0.101408
2,pchsale_pchinvt,0.022617,0.002138,0.093717
3,Vol_90D,0.022816,0.001939,0.084992
4,Vol_30D,0.023016,0.001740,0.076249
5,chcsho,0.023199,0.001556,0.068221
6,ep,0.023283,0.001472,0.064542
7,lev,0.023347,0.001408,0.061727
8,roeq,0.023386,0.001369,0.060019
9,me,0.023628,0.001127,0.049419


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.125403,"[64, 32, 16, 8]",0.0,0.001,0.00001,73,1024,73
1,2,2013-03-31,2014-02-28,0.125307,"[64, 32, 16, 8]",0.1,0.001,0.00010,47,1024,47
2,3,2014-03-31,2015-02-28,0.121306,"[64, 32, 16, 8]",0.1,0.001,0.00010,26,1024,26
3,4,2015-03-31,2016-02-29,0.110640,"[64, 32, 16, 8]",0.0,0.001,0.00010,18,1024,18
4,5,2016-03-31,2017-02-28,0.107679,"[64, 32]",0.1,0.001,0.00010,15,1024,15
5,6,2017-03-31,2018-02-28,0.133373,"[64, 32, 16, 8]",0.1,0.001,0.00001,26,1024,26
6,7,2018-03-31,2019-02-28,0.142926,"[64, 32, 16, 8]",0.1,0.001,0.00010,38,1024,38
7,8,2019-03-31,2020-02-29,0.129701,"[64, 32, 16, 8]",0.0,0.001,0.00010,17,1024,17
8,9,2020-03-31,2021-02-28,0.123181,"[64, 32, 16]",0.1,0.001,0.00010,10,1024,10
9,10,2021-03-31,2022-02-28,0.151839,"[64, 32, 16]",0.0,0.001,0.00010,32,1024,32


In [57]:
show_latest_recommendations('NN', feature_profile='careful_v3', top_k=10, save_to_run_dir=True)

Feature profile: careful_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,BCG VM Equity,2026-02-28,0.049847,2530.0000,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
1,2,BCE VM Equity,2026-02-28,0.048685,11350.0000,3.972500e+05,3.774800e+08,505.714286,-0.034043,0.036530,0.096618,0.975866
2,3,DTG VH Equity,2026-02-28,0.047712,146000.8336,1.446552e+05,2.403949e+09,1.090418,-0.044025,0.041096,-0.219654,1.374816
3,4,ASP VM Equity,2026-02-28,0.047611,5020.0000,1.874445e+05,9.516200e+07,8457.522508,0.070362,0.026585,-0.029014,1.722117
4,5,TCD VM Equity,2026-02-28,0.045860,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
5,6,GDT VM Equity,2026-02-28,0.042484,492184.2755,5.456466e+05,1.373137e+10,1.382804,0.010127,-0.036232,-0.143410,0.652403
6,7,DXP VH Equity,2026-02-28,0.040187,772840.7157,4.770080e+05,1.924284e+10,2.363206,0.102564,0.194444,0.121739,1.986881
7,8,TDC VM Equity,2026-02-28,0.039848,11500.0000,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
8,9,ST8 VM Equity,2026-02-28,0.039730,4000.0000,1.028836e+05,4.462470e+08,883.875584,-0.101124,-0.377916,-0.537572,2.801123
9,10,VNE VM Equity,2026-02-28,0.038905,430789.9733,1.881714e+06,8.283494e+07,0.033383,-0.067496,-0.177116,0.671975,0.434016


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260311T021101Z/recommendations/nn_careful_v3_latest_recommendations.csv


### max_v3

In [58]:
RUN_DIR_NN_MAX = run_single_model('NN', feature_profile='max_v3')
RUN_DIR_NN_MAX

Running: /usr/bin/python3 /content/drive/MyDrive/v2/model/run_model.py --config /content/drive/MyDrive/v2/model/configs/max_v3.yaml --models NN --stages all
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out = df.groupby('id', sort=False)[col].pct_change(periods)
/content/drive/MyDrive/v2/model/src/v2_model/prepare_inputs.py:32: FutureWarning:

PosixPath('/content/drive/MyDrive/v2/model/outputs/run_20260311T023211Z')

In [59]:
show_model_summary('NN', feature_profile='max_v3')

Feature profile: max_v3
Run dir: /content/drive/MyDrive/v2/model/outputs/run_20260311T023211Z


,metric,value
0,n_rows,108605
1,n_assets,699
2,n_months,270
3,n_optional_features,131
4,date_min,2003-10-31 00:00:00
5,date_max,2026-03-31 00:00:00


,metric,value
0,n_windows,15
1,first_train_range,2003-10-31 00:00:00 -> 2008-09-30 00:00:00
2,first_val_range,2008-10-31 00:00:00 -> 2010-09-30 00:00:00
3,first_test_range,2010-10-31 00:00:00 -> 2011-09-30 00:00:00
4,test_union_range,2010-10-31 00:00:00 -> 2025-09-30 00:00:00


,step,value
0,initial_rows,108605
1,after_price_size_filter,23116
2,after_liquidity_filter,16158
3,rows_with_known_target,16042
4,columns_kept_for_profile,140
5,requested_optional_low_coverage,6
6,after_monthly_median_fill_dropna,16042
7,training_rows,15871
8,n_features,134
9,feature_profile,max_v3


,sample,R2OOS_NN,n_rows,n_months
0,Full sample,-0.037840,12716,156
1,Large firms,-0.086102,3756,156
2,Small firms,-0.048539,3756,156


,strategy,mean_turnover,cost_bps,n_periods,mean_period,std_period,ann_ret,ann_vol,sharpe,cum_ret,max_dd,max_1m_loss
0,NN_LS_EW_gross,1.800096,0,156,0.030837,0.070296,0.401089,0.243512,1.519613,79.177857,-0.329049,-0.111882
1,NN_LS_EW_net_0bps,1.800096,0,156,0.030837,0.070296,0.401089,0.243512,1.519613,79.177857,-0.329049,-0.111882
2,NN_LS_EW_net_10bps,1.800096,10,156,0.029037,0.070180,0.372004,0.243109,1.433280,60.040628,-0.340527,-0.113448
3,NN_LS_EW_net_20bps,1.800096,20,156,0.027237,0.070071,0.343465,0.242732,1.346516,45.445303,-0.351836,-0.116751
4,NN_LS_EW_net_30bps,1.800096,30,156,0.025437,0.069969,0.315463,0.242381,1.259347,34.320086,-0.362978,-0.120054
5,NN_LS_VW_gross,2.250876,0,156,0.028153,0.147549,0.272737,0.511126,0.660954,21.993334,-0.662923,-0.184132
6,NN_LS_VW_net_0bps,2.250876,0,156,0.028153,0.147549,0.272737,0.511126,0.660954,21.993334,-0.662923,-0.184132
7,NN_LS_VW_net_10bps,2.250876,10,156,0.025902,0.147550,0.239251,0.511127,0.608108,15.258070,-0.694557,-0.185161
8,NN_LS_VW_net_20bps,2.250876,20,156,0.023651,0.147557,0.206559,0.511151,0.555237,10.484978,-0.723301,-0.186191
9,NN_LS_VW_net_30bps,2.250876,30,156,0.021400,0.147570,0.174644,0.511198,0.502348,7.105562,-0.750868,-0.187221


,weighting,strategy,benchmark,n_months,mean_active_return,information_ratio,corr_strategy_benchmark
0,equal,NN_LS_EW_net_30bps,benchmark_equal_excess,156,0.017776,0.205718,0.087473
1,value,NN_LS_VW_net_30bps,benchmark_value_excess,156,0.013739,0.088939,0.070172


,Feature,R2OOS,red_R2OOS,var_imp
0,ev_ebitda_raw,0.018187,-0.001360,0.044546
1,std_turn,0.018004,-0.001177,0.038556
2,idiovol,0.017782,-0.000955,0.031294
3,cfp_x_lev,0.017771,-0.000944,0.030917
4,opercf_assets_raw,0.017763,-0.000936,0.030673
5,mom1m_x_idiovol,0.017740,-0.000913,0.029917
6,ebitda_growth_12m_raw,0.017722,-0.000895,0.029328
7,grossprofit_assets_raw,0.017551,-0.000724,0.023713
8,intraday_range_vol_1m,0.017546,-0.000719,0.023541
9,me,0.017531,-0.000704,0.023054


,window_id,test_start,test_end,best_score,best_hidden_layers,best_dropout,best_learning_rate,best_weight_decay,best_best_epoch,best_batch_size,best_epoch
0,1,2012-03-31,2013-02-28,0.125496,"[64, 32, 16, 8]",0.0,0.001,0.00001,27,1024,27
1,2,2013-03-31,2014-02-28,0.125658,"[64, 32, 16, 8]",0.1,0.001,0.00010,22,1024,22
2,3,2014-03-31,2015-02-28,0.121683,"[64, 32, 16, 8]",0.1,0.001,0.00010,6,1024,6
3,4,2015-03-31,2016-02-29,0.111179,"[64, 32, 16, 8]",0.0,0.001,0.00001,4,1024,4
4,5,2016-03-31,2017-02-28,0.107592,"[64, 32, 16, 8]",0.1,0.001,0.00010,10,1024,10
5,6,2017-03-31,2018-02-28,0.132937,"[64, 32, 16, 8]",0.1,0.001,0.00010,14,1024,14
6,7,2018-03-31,2019-02-28,0.142902,"[64, 32, 16, 8]",0.1,0.001,0.00010,18,1024,18
7,8,2019-03-31,2020-02-29,0.129548,"[64, 32, 16, 8]",0.0,0.001,0.00010,6,1024,6
8,9,2020-03-31,2021-02-28,0.123972,"[64, 32, 16, 8]",0.0,0.001,0.00001,5,1024,5
9,10,2021-03-31,2022-02-28,0.152387,"[64, 32, 16, 8]",0.1,0.001,0.00001,14,1024,14


In [60]:
show_latest_recommendations('NN', feature_profile='max_v3', top_k=10, save_to_run_dir=True)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4782: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=l

Feature profile: max_v3
Latest fully labeled month scored: 2026-02-28
Calibration window train: 2017-03-31 -> 2022-02-28
Calibration window val  : 2022-03-31 -> 2024-02-29


,rank,id,eom,yhat_next_period,prc,me,adv_med,turn,mom1m,mom6m,ret_12_1,be_me
0,1,BAB VH Equity,2026-02-28,0.025255,12000.0000,1.286568e+07,1.146298e+08,8.636930,-0.035759,-0.139302,0.069358,1.034450
1,2,BCC VH Equity,2026-02-28,0.024876,7800.0000,9.610365e+05,5.184796e+08,254.922883,-0.025000,-0.071429,-0.037037,1.918663
2,3,ADS VM Equity,2026-02-28,0.024858,8400.0000,6.417157e+05,4.728175e+08,640.096509,0.021898,-0.015240,-0.133127,1.510058
3,4,FIR VM Equity,2026-02-28,0.023026,438858.3402,6.322401e+05,1.531971e+09,0.509506,-0.133891,-0.267689,-0.014288,1.177755
4,5,BCG VM Equity,2026-02-28,0.021065,2530.0000,2.226933e+06,3.696279e+10,16598.072480,0.000000,-0.287324,-0.589286,4.469659
5,6,TCD VM Equity,2026-02-28,0.021061,1890.0000,6.347010e+05,5.261382e+09,51990.594704,0.000000,-0.267442,-0.590909,5.903388
6,7,FDC VM Equity,2026-02-28,0.020912,714534.5280,6.507987e+05,1.187486e+11,10.322374,0.149068,0.149068,0.174603,1.021762
7,8,TD6 VH Equity,2026-02-28,0.019925,7200.0000,4.459335e+05,2.285637e+08,1245.173497,-0.052632,-0.172414,NaN,1.696055
8,9,TDC VM Equity,2026-02-28,0.019809,11500.0000,1.463122e+06,1.747538e+09,3825.013389,-0.008621,-0.068826,-0.132075,1.069443
9,10,GDT VM Equity,2026-02-28,0.018644,492184.2755,5.456466e+05,1.373137e+10,1.382804,0.010127,-0.036232,-0.143410,0.652403


Saved: /content/drive/MyDrive/v2/model/outputs/run_20260311T023211Z/recommendations/nn_max_v3_latest_recommendations.csv
